# 0. Optional Steps

The steps in this section are optional and used for debug logging

## 0.1 Set the environment to use a proxy

In [ ]:
import os

# Setup proxy through local proxy; bypass proxy for metadata endpoint to avoid any issues with Microsoft authentication libraries
os.environ["HTTP_PROXY"] = "http://127.0.0.1:8000"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:8000"
os.environ["NO_PROXY"] = "169.254.169.254"

# Add private CA root cert to trusted cert bundle for SDKs that use requests and httpx
os.environ["REQUESTS_CA_BUNDLE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"
os.environ["SSL_CERT_FILE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"

import os

# Setup proxy through local proxy; bypass proxy for metadata endpoint to avoid any issues with Microsoft authentication libraries
os.environ["HTTP_PROXY"] = "http://127.0.0.1:8000"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:8000"
os.environ["NO_PROXY"] = "169.254.169.254"

# Add private CA root cert to trusted cert bundle for SDKs that use requests and httpx
os.environ["REQUESTS_CA_BUNDLE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"
os.environ["SSL_CERT_FILE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"

# 1. Setup the notebook

The steps in this section setup the notebook

## 1.1 Install required libraries

In [ ]:
%pip install -r requirements.txt

## 1.2 Setup a logger

In [ ]:
# Setup a Python logger
import logging
import sys

def configure_logging(level="ERROR"):
    """This function configures logging for code being run based on the specified level.
    Args:
        level (str): The logging level to use (e.g., "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL").
    """

    try:
        # Convert the level string to uppercase so it matches what the logging library expects
        logging_level = getattr(logging, level.upper(), None)

        # Setup a logging format
        logging.basicConfig(
            level=logging_level,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[logging.StreamHandler(sys.stdout)]
        )
    except Exception as e:
        print(f"Failed to set up logging: {e}", file=sys.stderr)
        sys.exit(1) 

configure_logging("DEBUG")

## 1.3 Load environment variables

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(".env")

FOUNDRY_MODEL_ENDPOINT = os.getenv("FOUNDRY_MODEL_ENDPOINT")
ENTRA_TENANT_ID = os.getenv("ENTRA_TENANT_ID")
BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID = os.getenv("BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID")
BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET = os.getenv("BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET")
BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID = os.getenv("BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID")
BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET = os.getenv("BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET")
LLM_DEPLOYMENT_NAME = os.getenv("LLM_DEPLOYMENT_NAME")


# 2. Perform a single chat completion

In [ ]:
# Get an Entra token

from azure.identity import ClientSecretCredential, get_bearer_token_provider

credential = ClientSecretCredential(
    client_id=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID,
    client_secret=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET,
    tenant_id=ENTRA_TENANT_ID
)

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default"
)
print(token_provider())


In [ ]:
from openai import OpenAI


client = OpenAI(
    base_url=FOUNDRY_MODEL_ENDPOINT,
    api_key=token_provider
)

print(client.base_url)
prompt = "What is the capital of Sweden?"

completion = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
)
print(f"User: {prompt}")
print(f"Assistant: {completion.choices[0].message.content}")